<a href="https://colab.research.google.com/github/HST0077/HYOTC/blob/main/Quanto_FX_hedging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Plain Vanilla**

S&P 500 지수는 현재 6,800이고 변동성은 21%이다. 원달러 환율은 1,490 KRW/USD이고, 변동성은 10%이며, S&P 500지수와 환율의 상관관계는 (-)0.6 이라고 한다. 1년만기 행사가 6,800 콜옵션에 대하여 만기 환율을 현재 환율로 고정하여 지급하는 구조라고 할 때, 이 상품의 콜옵션 가격과 환율에 대한 민감도를 구하여 보아라.

In [1]:
"""
Black-Scholes 해석 공식 (배당수익률 q 적용)
Plain/BSM_MC.py의 MC_Call, MC_Greeks와 동일 계약·Greeks 정의
"""

from math import log, sqrt, exp
from scipy.stats import norm
from scipy.optimize import brentq


def _d1_d2(S, K, T, r, q, sigma):
    if T <= 0:
        return 0.0, 0.0
    sqrtT = sqrt(T)
    d1 = (log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    return d1, d2


def BSM(S, K, T, r, q, sigma, option="call"):
    """
    European 옵션 해석 가격 (연속 배당수익률 q).
    option: "call" | "put"
    """
    opt = option.lower().strip()
    if opt not in ("call", "put"):
        raise ValueError('option must be "call" or "put"')
    if T <= 0:
        return max(S - K, 0.0) if opt == "call" else max(K - S, 0.0)
    d1, d2 = _d1_d2(S, K, T, r, q, sigma)
    if opt == "call":
        return S * exp(-q * T) * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
    return K * exp(-r * T) * norm.cdf(-d2) - S * exp(-q * T) * norm.cdf(-d1)


In [2]:
"""
BSM Quanto valuation utilities.

가정:
- 기초는 해외지수 S (예: S&P500)
- 지급통화는 KRW (고정 환산/quanto 구조)
- 국내(원화) 측도에서 quanto drift 보정:
    (이미지 공식 기준, domestic measure)

    C = e^{-r_d T} [ S0 * e^{(r_f - q - ρ σ_S σ_X)T} N(d1) - K N(d2) ]
    P = e^{-r_d T} [ K N(-d2) - S0 * e^{(r_f - q - ρ σ_S σ_X)T} N(-d1) ]

    d1 = ( ln(S0/K) + (r_f - q - ρ σ_S σ_X + 0.5 σ_S^2)T ) / (σ_S √T)
    d2 = d1 - σ_S √T

  구현은 BSM 표준형으로 변환하여 사용:
    q_eff = r_d - (r_f - q - ρ σ_S σ_X) = r_d - r_f + q + ρ σ_S σ_X
"""

from math import exp, log, sqrt
from scipy.stats import norm


def _d1_d2_spot(S, K, T, sigma, drift_like):
    """lognormal with drift_like in d1 numerator: ln(S/K)+(drift_like+0.5*sigma^2)T."""
    if T <= 0:
        return 0.0, 0.0
    v = sigma * sqrt(T)
    d1 = (log(S / K) + (drift_like + 0.5 * sigma * sigma) * T) / v
    d2 = d1 - v
    return d1, d2


def _q_eff(r_d, r_f, q, rho_s_fx, sigma_s, sigma_fx):
    # q_eff = r_d - r_f + q + ρ σ_S σ_X
    return (r_d - r_f) + q + rho_s_fx * sigma_s * sigma_fx


def _d1_d2(S, K, T, r, q_adj, sigma):
    if T <= 0:
        return 0.0, 0.0
    sqrtT = sqrt(T)
    d1 = (log(S / K) + (r - q_adj + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    return d1, d2


def BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx, rho_s_fx, option="call"):
    """
    Quanto 조정(q_adj)을 적용한 유럽형 vanilla 옵션 해석 가격.

    이미지 공식의 quanto drift를 반영한 유럽형 vanilla 옵션 해석 가격.

    - r_krw: domestic rate (= r_d)
    - r_f: foreign risk-free rate
    - q: dividend yield on underlying
    - option: "call" | "put"
    """
    opt = option.lower().strip()
    if opt not in ("call", "put"):
        raise ValueError('option must be "call" or "put"')
    if T <= 0:
        return max(S - K, 0.0) if opt == "call" else max(K - S, 0.0)
    if S <= 0 or K <= 0 or sigma_s <= 0:
        raise ValueError("S, K, sigma_s must be positive")

    q_eff = _q_eff(r_krw, r_f, q, rho_s_fx, sigma_s, sigma_fx)
    d1, d2 = _d1_d2(S, K, T, r_krw, q_eff, sigma_s)
    if opt == "call":
        return S * exp(-q_eff * T) * norm.cdf(d1) - K * exp(-r_krw * T) * norm.cdf(d2)
    return K * exp(-r_krw * T) * norm.cdf(-d2) - S * exp(-q_eff * T) * norm.cdf(-d1)


def BSM_Quanto_Greeks(
    S,
    K,
    T,
    r_krw,
    r_f,
    q,
    sigma_s,
    sigma_fx,
    rho_s_fx,
    option="call",
    bump_sigma_fx_frac=0.01,
    bump_rho=0.005,
):
    """
    Quanto vanilla 옵션 해석적 Greeks.

    반환:
    - Price, Delta, Gamma, Theta(1day), Vega(1%) : 지수 변동성 σ_S 기준 (기존과 동일)
    - Rho(1bp) : 국내 금리 r_d 1bp
    - Vega_FX(1%) : FX 변동성 σ_X 1% (상대 bump, Vega와 동일 스케일)
    - Corr(1pt) : ρ(S,FX) 가 0.01 절대값 변할 때 가격 변화(중앙차분 × 0.01)
    - Rho_f(1bp) : 해외 무위험금리 r_f 1bp
    """
    opt = option.lower().strip()
    if opt not in ("call", "put"):
        raise ValueError('option must be "call" or "put"')

    if T <= 0:
        intrinsic = max(S - K, 0.0) if opt == "call" else max(K - S, 0.0)
        delta_int = (1.0 if S > K else 0.0) if opt == "call" else (-1.0 if S < K else 0.0)
        return {
            "Price": float(intrinsic),
            "Delta": float(delta_int),
            "Gamma": 0.0,
            "Theta": 0.0,
            "Vega": 0.0,
            "Rho": 0.0,
            "Vega_FX(1%)": 0.0,
            "Corr(1pt)": 0.0,
            "Rho_f(1bp)": 0.0,
        }

    if S <= 0 or K <= 0 or sigma_s <= 0:
        raise ValueError("S, K, sigma_s must be positive")

    q_eff = _q_eff(r_krw, r_f, q, rho_s_fx, sigma_s, sigma_fx)
    d1, d2 = _d1_d2(S, K, T, r_krw, q_eff, sigma_s)
    sqrtT = sqrt(T)
    n_d1 = norm.pdf(d1)

    price = BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx, rho_s_fx, opt)

    if opt == "call":
        Delta = exp(-q_eff * T) * norm.cdf(d1)
    else:
        Delta = -exp(-q_eff * T) * norm.cdf(-d1)

    Gamma = exp(-q_eff * T) * n_d1 / (S * sigma_s * sqrtT) if sqrtT > 0 and sigma_s > 0 else 0.0
    Vega = 0.01 * S * exp(-q_eff * T) * sqrtT * n_d1

    # Rho (1bp): bump r_krw
    dr = 0.0001
    price_up = BSM_Quanto(S, K, T, r_krw + dr, r_f, q, sigma_s, sigma_fx, rho_s_fx, opt)
    Rho = float(price_up - price)

    # Theta (1day): consistent with Plain/BSM.py convention
    eps_t = 1.0 / 365.0
    T_eps = max(T - eps_t, 1e-10)
    Theta = BSM_Quanto(S, K, T_eps, r_krw, r_f, q, sigma_s, sigma_fx, rho_s_fx, opt) - price

    # FX 변동성 민감도 (Vega_FX): σ_X에 대한 Vega와 동일 스케일(1% 상대 bump)
    h_fx = max(abs(sigma_fx) * bump_sigma_fx_frac, 1e-12)
    p_fxp = BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx + h_fx, rho_s_fx, opt)
    p_fxm = BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, max(sigma_fx - h_fx, 0.0), rho_s_fx, opt)
    Vega_FX = 0.01 * (p_fxp - p_fxm) / (2.0 * h_fx)

    # S–FX 상관 민감도: ρ 가 0.01 변할 때 가격 변화(ρ는 [-1,1]로 클램프)
    hr = float(bump_rho)
    rp = min(1.0, rho_s_fx + hr)
    rm = max(-1.0, rho_s_fx - hr)
    p_rp = BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx, rp, opt)
    p_rm = BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx, rm, opt)
    denom_rho = rp - rm
    Corr_1pt = (
        0.01 * (p_rp - p_rm) / denom_rho
        if denom_rho > 1e-15
        else 0.0
    )

    # 해외 금리 1bp
    drf = 0.0001
    p_rf_p = BSM_Quanto(S, K, T, r_krw, r_f + drf, q, sigma_s, sigma_fx, rho_s_fx, opt)
    p_rf_m = BSM_Quanto(S, K, T, r_krw, r_f - drf, q, sigma_s, sigma_fx, rho_s_fx, opt)
    Rho_f = 0.0001 * (p_rf_p - p_rf_m) / (2.0 * drf)

    return {
        "Price": float(price),
        "Delta": float(Delta),
        "Gamma": float(Gamma),
        "Theta": float(Theta),
        "Vega": float(Vega),
        "Rho": float(Rho),
        "Vega_FX(1%)": float(Vega_FX),
        "Corr(1pt)": float(Corr_1pt),
        "Rho_f(1bp)": float(Rho_f),
    }


In [3]:
S=6800
K=6800
T=1
r_krw=0.035
r_f=0.05
q=0.01
sigma_s=0.21
sigma_fx=0.1
rho_s_fx=-0.6

BSM_Quanto_Greeks(
    S,
    K,
    T,
    r_krw,
    r_f,
    q,
    sigma_s,
    sigma_fx,
    rho_s_fx,
    option="call",
    bump_sigma_fx_frac=0.01,
    bump_rho=0.005,
)

{'Price': 758.7529704343142,
 'Delta': 0.6502274445468526,
 'Gamma': 0.00026692299828300466,
 'Theta': -1.3105680481353374,
 'Vega': 25.919290825272885,
 'Rho': -0.07587150340486915,
 'Vega_FX(1%)': 5.571148772262404,
 'Corr(1pt)': -0.9285247939833418,
 'Rho_f(1bp)': 0.4421546636608582}

In [4]:
callprice=BSM(S, K, T, r_f, q, sigma_s, option="call")
Quanto_callprice=BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx, rho_s_fx, option="call")
print("Call Price without Quanto:",callprice)
print("Call Price with Quanto:",Quanto_callprice)

Call Price without Quanto: 693.8825051448948
Call Price with Quanto: 758.7529704343142


In [5]:
price=BSM(S, K, T, r_f, q, sigma_s, option="put")
Quanto_price=BSM_Quanto(S, K, T, r_krw, r_f, q, sigma_s, sigma_fx, rho_s_fx, option="put")
print("Put Price without Quanto:",price)
print("PUt Price with Quanto:",Quanto_price)

Put Price without Quanto: 429.90372225540705
PUt Price with Quanto: 404.1304110241331


앞 문제의 콜옵션을 매수하였다고 하고, 하루 후 주가가 10% 상승,하락하고, 환율이 각각 10% 상승, 하락한 경우의 KRW 기준 P&L을 작성해 보아라.

In [6]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def bsm_quanto_info(S, K, T, r_d, r_f, q, sigma_s, sigma_fx, rho):
    if T <= 0: return max(S - K, 0.0), (1.0 if S > K else 0.0)
    drift = r_f - q - rho * sigma_s * sigma_fx
    d1 = (np.log(S / K) + (drift + 0.5 * sigma_s**2) * T) / (sigma_s * np.sqrt(T))
    d2 = d1 - sigma_s * np.sqrt(T)

    price_pts = np.exp(-r_d * T) * (S * np.exp(drift * T) * norm.cdf(d1) - K * norm.cdf(d2))
    delta = np.exp((drift - r_d) * T) * norm.cdf(d1)
    return price_pts, delta

# 1. 초기 파라미터 (문제 1 데이터)
S0, K, T0 = 6800, 6800, 1.0
r_d, r_f, q = 0.035, 0.05, 0.01
sigma_s, sigma_fx, rho = 0.21, 0.10, -0.6
X_fix = 1490

# [Day 0] 초기 시점
p0_usd, delta0 = bsm_quanto_info(S0, K, T0, r_d, r_f, q, sigma_s, sigma_fx, rho)
initial_opt_krw = p0_usd * X_fix

# 2. [Day 1] 시나리오 (하루 후)
T1 = T0 - (1/365)
price_moves = [0.9, 1.1]
fx_moves = [0.9, 1.1]

scenario_results = []

for p_m in price_moves:
    S1 = S0 * p_m
    p1_usd, _ = bsm_quanto_info(S1, K, T1, r_d, r_f, q, sigma_s, sigma_fx, rho)

    # [KRW 기준 옵션 손익 - 환율과 무관하게 고정환율 적용]
    opt_pnl_krw = (p1_usd * X_fix) - initial_opt_krw

    # [USD 기준 델타 손익 - 주식 공매도 포지션]
    delta_pnl_usd = -(S1 - S0) * delta0

    for f_m in fx_moves:
        X1 = X_fix * f_m

        # [KRW 기준 최종 P&L]
        # 1. 환헤지 시 (Hedged): 주식 손익도 고정환율 1490으로 확정
        krw_pl_hedged = opt_pnl_krw + (delta_pnl_usd * X_fix)

        # 2. 환미헤지 시 (Unhedged): 주식 숏 포지션의 외화가치가 시장환율 X1에 노출
        # (매도시점 원화가치) - (현재시점 상환 원화가치) + 옵션 손익(KRW)
        delta_pnl_krw_unhedged = (S0 * delta0 * X_fix) - (S1 * delta0 * X1)
        krw_pl_unhedged = opt_pnl_krw + delta_pnl_krw_unhedged

        scenario_results.append({
            "주가변동": f"{(p_m-1)*100:+.0f}%",
            "환율변동": f"{(f_m-1)*100:+.0f}%",
            "Option P&L (KRW)": int(opt_pnl_krw),
            "Delta P&L ($)": round(delta_pnl_usd, 2),
            "KRW P&L (Hedged)": int(krw_pl_hedged),
            "KRW P&L (Unhedged)": int(krw_pl_unhedged)
        })

# 3. DataFrame 출력
df_final = pd.DataFrame(scenario_results)
print(" KRW 성과 중심 시나리오 분석 ")
df_final

 KRW 성과 중심 시나리오 분석 


,주가변동,환율변동,Option P&L (KRW),Delta P&L ($),KRW P&L (Hedged),KRW P&L (Unhedged)
0,-10%,-10%,-561491,442.15,97319,690248
1,-10%,+10%,-561491,442.15,97319,-495610
2,+10%,-10%,739776,-442.15,80965,805657
3,+10%,+10%,739776,-442.15,80965,-643725


In [7]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# 1. 퀀토 옵션 가격 및 델타 계산 함수 (이미지 공식 기반)
def get_quanto_info(S, K, T, r_d, r_f, q, sigma_s, sigma_fx, rho):
    if T <= 1e-6: # 만기 시점
        return max(S - K, 0.0), (1.0 if S > K else 0.0)

    drift = r_f - q - rho * sigma_s * sigma_fx
    d1 = (np.log(S / K) + (drift + 0.5 * sigma_s**2) * T) / (sigma_s * np.sqrt(T))
    d2 = d1 - sigma_s * np.sqrt(T)

    price_pts = np.exp(-r_d * T) * (S * np.exp(drift * T) * norm.cdf(d1) - K * norm.cdf(d2))
    delta = np.exp((drift - r_d) * T) * norm.cdf(d1)
    return price_pts, delta

# 2. 시뮬레이션 파라미터 설정
S0, K = 6800, 6800
T_total = 1.0        # 만기 1년
r_d, r_f, q = 0.035, 0.05, 0.01
sigma_s, sigma_fx, rho = 0.21, 0.10, -0.6
X_fix = 1490
steps = 252          # 리밸런싱 횟수 (영업일 기준)
dt = T_total / steps
num_fx_paths = 1000

# 3. 주가 경로 생성 (1개 고정)
np.random.seed(42) # 재현성을 위한 시드 설정
S_path = [S0]
for _ in range(steps):
    z = np.random.standard_normal()
    # 주가 성장률은 미국 금리(r_f)와 배당(q) 기준
    S_next = S_path[-1] * np.exp((r_f - q - 0.5 * sigma_s**2) * dt + sigma_s * np.sqrt(dt) * z)
    S_path.append(S_next)
S_path = np.array(S_path)

# 4. 환율 경로 시뮬레이션 및 헤지 수행
p0_pts, _ = get_quanto_info(S0, K, T_total, r_d, r_f, q, sigma_s, sigma_fx, rho)
initial_cost_krw = p0_pts * X_fix  # 초기 옵션 매수 비용

final_pnl_hedged = []
final_pnl_unhedged = []

for j in range(num_fx_paths):
    # 환율 경로 생성 (주가와 상관계수 rho 반영)
    # 실제 시장에선 주가와 상관관계가 있으나 여기선 FX 고유 변동성 중심으로 생성
    z_fx = np.random.standard_normal(steps)
    X_path = [X_fix]
    for i in range(steps):
        # 환율 성장률은 국내외 금리차 기준
        X_next = X_path[-1] * np.exp((r_d - r_f - 0.5 * sigma_fx**2) * dt + sigma_fx * np.sqrt(dt) * z_fx[i])
        X_path.append(X_next)
    X_path = np.array(X_path)

    # 헤지 시뮬레이션 초기화
    cash_hedged = -initial_cost_krw
    cash_unhedged = -initial_cost_krw
    curr_delta = 0

    # 매일 델타 리밸런싱 수행 (Long Call + Short Stock)
    for t in range(steps):
        t_rem = T_total - t * dt
        _, new_delta = get_quanto_info(S_path[t], K, t_rem, r_d, r_f, q, sigma_s, sigma_fx, rho)

        diff = new_delta - curr_delta # 주식 매도(-) 수량 변화

        # 1) 환헤지 시: 모든 주식 매매 대금(USD)을 X_fix(1490원)로 즉시 환전
        cash_hedged += diff * S_path[t] * X_fix

        # 2) 환미헤지 시: 주식 매매 시점의 시장 환율 X_path[t] 적용
        cash_unhedged += diff * S_path[t] * X_path[t]

        curr_delta = new_delta

    # 만기 정산 (Long Call 행사 + Short Stock 상환)
    payoff_krw = max(S_path[-1] - K, 0) * X_fix

    # 환헤지 시: 마지막 주식 상환도 X_fix로
    final_pnl_hedged.append(cash_hedged - curr_delta * S_path[-1] * X_fix + payoff_krw)

    # 환미헤지 시: 마지막 주식 상환은 만기 시점 환율 X_path[-1]로
    final_pnl_unhedged.append(cash_unhedged - curr_delta * S_path[-1] * X_path[-1] + payoff_krw)

# 5. 결과 정리 및 통계 표 출력
results_df = pd.DataFrame({
    'Hedged P&L (KRW)': final_pnl_hedged,
    'Unhedged P&L (KRW)': final_pnl_unhedged
})

summary_stats = results_df.describe().loc[['mean', 'std', 'min', 'max']]
print("--- 1,000개 FX 시나리오 기반 헤징 성과 분석 ---")
print(summary_stats.to_string(formatters={'Hedged P&L (KRW)': '{:,.0f}'.format, 'Unhedged P&L (KRW)': '{:,.0f}'.format}))

--- 1,000개 FX 시나리오 기반 헤징 성과 분석 ---
     Hedged P&L (KRW) Unhedged P&L (KRW)
mean         -219,503           -158,179
std                 0            383,877
min          -219,503         -1,379,873
max          -219,503            852,918


# **Quanto 1star Step Down **

In [8]:
# -*- coding: utf-8 -*-
"""
공통 백엔드 모듈: GPU(CuPy) / CPU(NumPy) 자동 감지 및 통합 인터페이스

- get_backend(): 배열 연산용 라이브러리(cupy 또는 numpy) 반환, 한 번만 감지 후 캐시
- init_backend(): 클라이언트 시작 시 명시적 초기화 (분산 환경 권장)
- get_capability(): "gpu" 또는 "cpu" 문자열 반환 (서버에 보고용)
"""

_XP = None
_CAPABILITY = None


def init_backend():
    """
    백엔드를 초기화하고 반환.
    분산 클라이언트 시작 시 한 번 호출 권장.
    """
    global _XP, _CAPABILITY
    if _XP is not None:
        return _XP

    try:
        import cupy as cp
        _ = cp.array(1.0) + 1
        _XP = cp
        _CAPABILITY = "gpu"
    except Exception:
        import numpy as np
        _XP = np
        _CAPABILITY = "cpu"
    return _XP


def get_backend():
    """배열 연산 백엔드 반환 (lazy init: 아직 안 했으면 이 시점에 감지)"""
    global _XP, _CAPABILITY
    if _XP is None:
        init_backend()
    return _XP


def get_capability():
    """'gpu' 또는 'cpu' 반환 (서버에 capability 보고용)"""
    global _CAPABILITY
    if _CAPABILITY is None:
        init_backend()
    return _CAPABILITY


def _to_float(x):
    """백엔드 결과를 Python float로 변환 (CuPy는 .get() 필요)"""
    if hasattr(x, "get"):
        return float(x.get())
    return float(x)


In [9]:
"""
SD_1star (1자산) StepDown ELS Monte Carlo — Quanto adjusted

SD_1star_MC.py와 동일한 상품 구조·CRN(Z) 규칙.
차이점: 해외지수 등 KRW 발행 시 drift에 quanto 보정을 반영.

이미지/BSM_Quanto.py와 동일한 quanto drift(국내측도):

    μ_q = r_f - q - I * rho_s_fx * sigma * sigma_fx
    일일 drift = (μ_q - 0.5 * sigma^2) * dt

할인(현가)은 domestic rate r(=KRW 무위험)로 동일.

- r: KRW 무위험 이자율 (할인에도 동일 사용)
- r_f: 해외 무위험 이자율 (quanto drift에 사용). None이면 r_f=r로 처리(하위호환)
- sigma_fx: 지수 통화 / KRW FX 변동성
- rho_s_fx: Corr(dW_S, dW_FX)
- quanto_on=False 이면 μ_q = r_f - q 로 처리 (ρ-term 제거)

- SD_1star_MC_quanto: 통합 pricer (Z 외부 주입 CRN 지원)
- SD_1star_MC_batch_quanto: 복수 S0 일괄
- MC_Greeks_SD1_Quanto / MC_Greeks_SD1_Quanto_GPU_CRN: CRN Greeks
"""

import numpy as np

def SD_1star_MC_quanto(
    S0,
    kijun,
    K,
    T,
    c,
    r,
    q,
    sigma,
    barrier,
    dummy,
    sim,
    seed=111,
    antithetic=True,
    Z=None,
    return_Z=False,
    notional=10000.0,
    dtype="float64",
    sigma_fx=0.0,
    rho_s_fx=0.0,
    r_f=None,
    quanto_on=True,
):
    """
    SD_1star (1자산) StepDown ELS Monte Carlo with quanto drift adjustment.

    상품 구조는 SD_1star_MC와 동일:
    - R = S / kijun
    - 조기상환: T[i]일에 R >= K[i] → notional*(1+c[i]) 할인 지급
    - 만기: min(R) < barrier → 낙인, else notional*(1+dummy)
    """
    xp = get_backend()
    dtype_xp = xp.float64 if dtype == "float64" else xp.float32

    T_np = np.atleast_1d(np.asarray(T, dtype=int))
    K_np = np.atleast_1d(np.asarray(K, dtype=float))
    c_np = np.atleast_1d(np.asarray(c, dtype=float))

    if not (len(K_np) == len(T_np) == len(c_np)):
        raise ValueError("K, T, c 길이 불일치")

    K_arr = xp.asarray(K_np, dtype=dtype_xp)
    c_arr = xp.asarray(c_np, dtype=dtype_xp)

    sigma_xp = dtype_xp(sigma)
    q_xp = dtype_xp(q)
    r_f_xp = dtype_xp(r if r_f is None else r_f)
    sf = dtype_xp(sigma_fx)
    rsf = dtype_xp(rho_s_fx)
    mu_q = r_f_xp - q_xp - (dtype_xp(1.0) if quanto_on else dtype_xp(0.0)) * rsf * sigma_xp * sf

    # ---------- Z 준비 (CRN) — SD_1star_MC와 동일 ----------
    if Z is None:
        N = int(T_np[-1])
        if N <= 0:
            raise ValueError("T[-1] (만기일)은 양의 정수(일)여야 합니다.")
        xp.random.seed(seed)
        if antithetic:
            half = (sim + 1) // 2
            Z_half = xp.random.standard_normal((N, half))
            Z = xp.concatenate([Z_half, -Z_half], axis=1)[:, :sim]
        else:
            Z = xp.random.standard_normal((N, sim))
    else:
        Z = xp.asarray(Z).astype(dtype_xp, copy=False)
        if not (hasattr(Z, "shape") and len(Z.shape) == 2):
            raise ValueError("Z는 (N, sim) 형태여야 합니다.")
        N, sim_Z = Z.shape
        sim = int(sim_Z)
        if int(T_np[-1]) > N:
            raise ValueError(f"T[-1]={int(T_np[-1])}가 Z의 N={N}보다 큽니다.")

    dt = dtype_xp(1.0 / 365.0)
    drift = (mu_q - 0.5 * sigma_xp**2) * dt
    vol = sigma_xp * xp.sqrt(dt)

    lnS = drift + vol * Z
    lnS0 = xp.full((1, sim), xp.log(dtype_xp(S0)), dtype=dtype_xp)
    lnS = xp.concatenate([lnS0, lnS], axis=0)

    S_path = xp.exp(xp.cumsum(lnS, axis=0))
    R = S_path / dtype_xp(kijun)

    Price = xp.zeros(sim, dtype=dtype_xp)
    EN = len(T_np)
    r_xp = dtype_xp(r)

    for i in range(EN - 1):
        t = int(T_np[i])
        alive = Price == 0.0
        hit = alive & (R[t, :] >= K_arr[i])
        Price = xp.where(
            hit,
            dtype_xp(notional) * (dtype_xp(1.0) + c_arr[i]) * xp.exp(-r_xp * t / 365.0),
            Price,
        )

    alive = Price == 0.0
    if xp.any(alive):
        R_alive = R[:, alive]
        ki_hit = xp.min(R_alive, axis=0) < dtype_xp(barrier)
        t_mat = int(T_np[-1])
        payoff_ki = dtype_xp(notional) * R_alive[t_mat, :] * xp.exp(-r_xp * t_mat / 365.0)
        payoff_no = dtype_xp(notional) * (dtype_xp(1.0) + dtype_xp(dummy)) * xp.exp(-r_xp * t_mat / 365.0)
        Price_alive = xp.where(ki_hit, payoff_ki, payoff_no)
        Price[alive] = Price_alive

    mu = _to_float(xp.mean(Price))
    if return_Z:
        return mu, Z
    return mu


SD_1star_MC_quanto_CPU = SD_1star_MC_quanto
SD_lstar_GPU_CRN_quanto = SD_1star_MC_quanto


def SD_1star_MC_batch_quanto(
    S0_arr,
    kijun,
    K,
    T,
    c,
    r,
    q,
    sigma,
    barrier,
    dummy,
    sim,
    seed=111,
    antithetic=True,
    Z=None,
    notional=10000.0,
    dtype="float64",
    sigma_fx=0.0,
    rho_s_fx=0.0,
    r_f=None,
    quanto_on=True,
):
    """복수 S0 일괄. G 경로 1회 + quanto drift 반영."""
    xp = get_backend()
    dtype_xp = xp.float64 if dtype == "float64" else xp.float32

    T_np = np.atleast_1d(np.asarray(T, dtype=int))
    K_np = np.atleast_1d(np.asarray(K, dtype=float))
    c_np = np.atleast_1d(np.asarray(c, dtype=float))
    if not (len(K_np) == len(T_np) == len(c_np)):
        raise ValueError("K, T, c 길이 불일치")

    K_arr = xp.asarray(K_np, dtype=dtype_xp)
    c_arr = xp.asarray(c_np, dtype=dtype_xp)
    S0_arr = np.atleast_1d(np.asarray(S0_arr, dtype=float))

    sigma_xp = dtype_xp(sigma)
    q_xp = dtype_xp(q)
    r_f_xp = dtype_xp(r if r_f is None else r_f)
    sf = dtype_xp(sigma_fx)
    rsf = dtype_xp(rho_s_fx)
    mu_q = r_f_xp - q_xp - (dtype_xp(1.0) if quanto_on else dtype_xp(0.0)) * rsf * sigma_xp * sf
    r_xp = dtype_xp(r)

    if Z is None:
        N = int(T_np[-1])
        if N <= 0:
            raise ValueError("T[-1] (만기일)은 양의 정수(일)여야 합니다.")
        xp.random.seed(seed)
        if antithetic:
            half = (sim + 1) // 2
            Z_half = xp.random.standard_normal((N, half))
            Z = xp.concatenate([Z_half, -Z_half], axis=1)[:, :sim]
        else:
            Z = xp.random.standard_normal((N, sim))
    else:
        Z = xp.asarray(Z).astype(dtype_xp, copy=False)
        N, sim_Z = Z.shape
        sim = int(sim_Z)
        if int(T_np[-1]) > N:
            raise ValueError(f"T[-1]={int(T_np[-1])}가 Z의 N={N}보다 큽니다.")

    dt = dtype_xp(1.0 / 365.0)
    drift = (mu_q - 0.5 * sigma_xp**2) * dt
    vol = sigma_xp * xp.sqrt(dt)

    daily_ln = drift + vol * Z
    lnG = xp.concatenate([xp.zeros((1, sim), dtype=dtype_xp), daily_ln], axis=0)
    G = xp.exp(xp.cumsum(lnG, axis=0))

    price_arr = np.zeros(len(S0_arr), dtype=float)
    EN = len(T_np)

    for i, S0 in enumerate(S0_arr):
        scale = dtype_xp(S0 / kijun)
        R = scale * G
        Price = xp.zeros(sim, dtype=dtype_xp)

        for j in range(EN - 1):
            t = int(T_np[j])
            alive = Price == 0.0
            hit = alive & (R[t, :] >= K_arr[j])
            Price = xp.where(
                hit,
                dtype_xp(notional) * (dtype_xp(1.0) + c_arr[j]) * xp.exp(-r_xp * t / 365.0),
                Price,
            )

        alive = Price == 0.0
        if xp.any(alive):
            R_alive = R[:, alive]
            ki_hit = xp.min(R_alive, axis=0) < dtype_xp(barrier)
            t_mat = int(T_np[-1])
            payoff_ki = dtype_xp(notional) * R_alive[t_mat, :] * xp.exp(-r_xp * t_mat / 365.0)
            payoff_no = dtype_xp(notional) * (dtype_xp(1.0) + dtype_xp(dummy)) * xp.exp(-r_xp * t_mat / 365.0)
            Price_alive = xp.where(ki_hit, payoff_ki, payoff_no)
            Price[alive] = Price_alive

        price_arr[i] = _to_float(xp.mean(Price))

    return price_arr


def MC_Greeks_SD1_Quanto(
    S0,
    kijun,
    K,
    T,
    c,
    r,
    q,
    sigma,
    barrier,
    dummy,
    sim,
    seed=111,
    antithetic=True,
    notional=10000.0,
    dtype="float64",
    pricer=None,
    bump_S_frac=0.01,
    bump_sigma_frac=0.01,
    bump_r_bp=1,
    sigma_fx=0.0,
    rho_s_fx=0.0,
    r_f=None,
    quanto_on=True,
    bump_sigma_fx_frac=0.01,
    bump_rho=0.005,
):
    """Quanto pricer 기반 CRN Greeks (단일 Gamma bump)."""
    if pricer is None:
        pricer = SD_1star_MC_quanto

    T = np.atleast_1d(np.asarray(T, dtype=int))
    q_kw = dict(sigma_fx=sigma_fx, rho_s_fx=rho_s_fx, quanto_on=quanto_on, r_f=r_f)

    C0, Z = pricer(
        S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
        sim=sim, seed=seed, antithetic=antithetic,
        Z=None, return_Z=True, notional=notional, dtype=dtype, **q_kw
    )

    eps_S = max(S0 * sigma * bump_S_frac, 1e-12)
    Cp = pricer(S0 + eps_S, kijun, K, T, c, r, q, sigma, barrier, dummy,
                sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Cm = pricer(S0 - eps_S, kijun, K, T, c, r, q, sigma, barrier, dummy,
                sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Delta = (Cp - Cm) / (2.0 * eps_S)
    Gamma = (Cp - 2.0 * C0 + Cm) / (eps_S**2)

    eps_sig = max(sigma * bump_sigma_frac, 1e-12)
    Csig_p = pricer(S0, kijun, K, T, c, r, q, sigma + eps_sig, barrier, dummy,
                    sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Csig_m = pricer(S0, kijun, K, T, c, r, q, max(sigma - eps_sig, 1e-8), barrier, dummy,
                    sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Vega_1pct = 0.01 * (Csig_p - Csig_m) / (2.0 * eps_sig)

    Tm = np.maximum(T - 1, 0)
    Ct = pricer(S0, kijun, K, Tm, c, r, q, sigma, barrier, dummy,
                sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Theta_1d = Ct - C0

    eps_r = bump_r_bp * 0.0001
    Cr_p = pricer(S0, kijun, K, T, c, r + eps_r, q, sigma, barrier, dummy,
                  sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Cr_m = pricer(S0, kijun, K, T, c, r - eps_r, q, sigma, barrier, dummy,
                  sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Rho_1bp = 0.0001 * (Cr_p - Cr_m) / (2.0 * eps_r)

    # FX 변동성 σ_X (1% 상대 bump, Vega(1%)와 동일 스케일)
    h_fx = max(float(sigma_fx) * bump_sigma_fx_frac, 1e-12)
    kw_fxp = {**q_kw, "sigma_fx": float(sigma_fx) + h_fx}
    kw_fxm = {**q_kw, "sigma_fx": max(float(sigma_fx) - h_fx, 0.0)}
    Cfx_p = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_fxp)
    Cfx_m = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_fxm)
    Vega_FX_1pct = 0.01 * (Cfx_p - Cfx_m) / (2.0 * h_fx)

    # S–FX 상관: ρ 가 0.01 변할 때 가격 변화
    hr = float(bump_rho)
    rp = min(1.0, float(rho_s_fx) + hr)
    rm = max(-1.0, float(rho_s_fx) - hr)
    kw_rp = {**q_kw, "rho_s_fx": rp}
    kw_rm = {**q_kw, "rho_s_fx": rm}
    Crp = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                 sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rp)
    Crm = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                 sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rm)
    denom_rho = rp - rm
    Corr_1pt = 0.01 * (Crp - Crm) / denom_rho if denom_rho > 1e-15 else 0.0

    # 해외 금리 r_f 1bp
    drf = 0.0001
    base_rf = r if r_f is None else float(r_f)
    kw_rfp = {**q_kw, "r_f": base_rf + drf}
    kw_rfm = {**q_kw, "r_f": base_rf - drf}
    Crf_p = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rfp)
    Crf_m = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rfm)
    Rho_f_1bp = 0.0001 * (Crf_p - Crf_m) / (2.0 * drf)

    return {
        "Price": C0,
        "Delta": Delta,
        "Gamma": Gamma,
        "Vega(1%)": Vega_1pct,
        "Theta(1d)": Theta_1d,
        "Rho(1bp)": Rho_1bp,
        "Vega_FX(1%)": Vega_FX_1pct,
        "Corr(1pt)": Corr_1pt,
        "Rho_f(1bp)": Rho_f_1bp,
    }


def MC_Greeks_SD1_Quanto_GPU_CRN(
    S0,
    kijun,
    K,
    T,
    c,
    r,
    q,
    sigma,
    barrier,
    dummy,
    sim,
    seed=111,
    antithetic=True,
    notional=10000.0,
    dtype="float64",
    pricer=None,
    bump_S_frac=0.01,
    bump_sigma_frac=0.01,
    bump_r_bp=1,
    gamma_smooth=True,
    gamma_mults=(0.5, 1.0, 2.0, 4.0, 8.0),
    gamma_agg="median",
    gamma_trim=0.25,
    sigma_fx=0.0,
    rho_s_fx=0.0,
    r_f=None,
    quanto_on=True,
    bump_sigma_fx_frac=0.01,
    bump_rho=0.005,
):
    """Quanto pricer + Gamma 다점 스무딩."""
    if pricer is None:
        pricer = SD_1star_MC_quanto

    T = np.atleast_1d(np.asarray(T, dtype=int))
    q_kw = dict(sigma_fx=sigma_fx, rho_s_fx=rho_s_fx, quanto_on=quanto_on, r_f=r_f)

    C0, Z = pricer(
        S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
        sim=sim, seed=seed, antithetic=antithetic,
        Z=None, return_Z=True, notional=notional, dtype=dtype, **q_kw
    )

    eps_S_delta = max(S0 * sigma * bump_S_frac, 1e-12)
    Cp = pricer(S0 + eps_S_delta, kijun, K, T, c, r, q, sigma, barrier, dummy,
                sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Cm = pricer(S0 - eps_S_delta, kijun, K, T, c, r, q, sigma, barrier, dummy,
                sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Delta = (Cp - Cm) / (2.0 * eps_S_delta)

    if not gamma_smooth:
        Gamma = (Cp - 2.0 * C0 + Cm) / (eps_S_delta**2)
    else:
        eps_base = max(S0 * bump_S_frac, 1e-12)
        gammas = []
        for m in gamma_mults:
            eps = float(eps_base * m)
            if eps == 0:
                continue
            Cp_g = pricer(S0 + eps, kijun, K, T, c, r, q, sigma, barrier, dummy,
                          sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
            Cm_g = pricer(S0 - eps, kijun, K, T, c, r, q, sigma, barrier, dummy,
                          sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
            gammas.append((Cp_g - 2.0 * C0 + Cm_g) / (eps * eps))
        if len(gammas) == 0:
            Gamma = float("nan")
        else:
            gammas = np.asarray(gammas, dtype=float)
            if gamma_agg == "median":
                Gamma = float(np.median(gammas))
            elif gamma_agg == "trimmed_mean":
                trim = max(0.0, min(float(gamma_trim), 0.49))
                g_sorted = np.sort(gammas)
                n = len(g_sorted)
                k = int(np.floor(trim * n))
                core = g_sorted[k : n - k] if (n - 2 * k) >= 1 else g_sorted
                Gamma = float(np.mean(core))
            else:
                raise ValueError("gamma_agg는 'median' 또는 'trimmed_mean'만 지원합니다.")

    eps_sig = max(sigma * bump_sigma_frac, 1e-12)
    Csig_p = pricer(S0, kijun, K, T, c, r, q, sigma + eps_sig, barrier, dummy,
                    sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Csig_m = pricer(S0, kijun, K, T, c, r, q, sigma - eps_sig, barrier, dummy,
                    sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Vega_1pct = 0.01 * (Csig_p - Csig_m) / (2.0 * eps_sig)

    Tm = np.maximum(T - 1, 0)
    Ct = pricer(S0, kijun, K, Tm, c, r, q, sigma, barrier, dummy,
                sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Theta_1d = Ct - C0

    eps_r = bump_r_bp * 0.0001
    Cr_p = pricer(S0, kijun, K, T, c, r + eps_r, q, sigma, barrier, dummy,
                  sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Cr_m = pricer(S0, kijun, K, T, c, r - eps_r, q, sigma, barrier, dummy,
                  sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **q_kw)
    Rho_1bp = 0.0001 * (Cr_p - Cr_m) / (2.0 * eps_r)

    h_fx = max(float(sigma_fx) * bump_sigma_fx_frac, 1e-12)
    kw_fxp = {**q_kw, "sigma_fx": float(sigma_fx) + h_fx}
    kw_fxm = {**q_kw, "sigma_fx": max(float(sigma_fx) - h_fx, 0.0)}
    Cfx_p = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_fxp)
    Cfx_m = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_fxm)
    Vega_FX_1pct = 0.01 * (Cfx_p - Cfx_m) / (2.0 * h_fx)

    hr = float(bump_rho)
    rp = min(1.0, float(rho_s_fx) + hr)
    rm = max(-1.0, float(rho_s_fx) - hr)
    kw_rp = {**q_kw, "rho_s_fx": rp}
    kw_rm = {**q_kw, "rho_s_fx": rm}
    Crp = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                 sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rp)
    Crm = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                 sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rm)
    denom_rho = rp - rm
    Corr_1pt = 0.01 * (Crp - Crm) / denom_rho if denom_rho > 1e-15 else 0.0

    drf = 0.0001
    base_rf = r if r_f is None else float(r_f)
    kw_rfp = {**q_kw, "r_f": base_rf + drf}
    kw_rfm = {**q_kw, "r_f": base_rf - drf}
    Crf_p = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rfp)
    Crf_m = pricer(S0, kijun, K, T, c, r, q, sigma, barrier, dummy,
                   sim=sim, Z=Z, return_Z=False, notional=notional, dtype=dtype, **kw_rfm)
    Rho_f_1bp = 0.0001 * (Crf_p - Crf_m) / (2.0 * drf)

    return {
        "Price": C0,
        "Delta": Delta,
        "Gamma": Gamma,
        "Vega(1%)": Vega_1pct,
        "Theta(1d)": Theta_1d,
        "Rho(1bp)": Rho_1bp,
        "Vega_FX(1%)": Vega_FX_1pct,
        "Corr(1pt)": Corr_1pt,
        "Rho_f(1bp)": Rho_f_1bp,
        "bump": {
            "eps_S_delta": eps_S_delta,
            "eps_S_gamma_base": (S0 * bump_S_frac) if gamma_smooth else eps_S_delta,
            "eps_sigma": eps_sig,
            "eps_r": eps_r,
            "CRN": True,
            "antithetic": antithetic,
            "seed": seed,
            "gamma_smooth": gamma_smooth,
            "gamma_mults": tuple(gamma_mults),
            "gamma_agg": gamma_agg,
            "quanto": {
                "sigma_fx": sigma_fx,
                "rho_s_fx": rho_s_fx,
                "quanto_on": quanto_on,
                "bump_sigma_fx_frac": bump_sigma_fx_frac,
                "bump_rho": bump_rho,
            },
        },
    }


MC_Greeks_SD1_Quanto_CRN = MC_Greeks_SD1_Quanto_GPU_CRN

In [13]:
# 환 변동성이 0인 경우
# 스텝다운 상품 평가 정보
S0=6800
kijun=6800
r=0.035
r_f=0.05
q=0
sigma, barrier, dummy, sim=0.3, 0.65, 0.132,10000
K=[0.95,0.95,0.95,0.90,0.90,0.85]
T=np.array([ 181,  365,  546,  730,  912, 1098])
c=np.array([1,2,3,4,5,6])*0.022
sigma_fx=0.1
rho_s_fx=0

MC_Greeks_SD1_Quanto_GPU_CRN(
    S0,
    kijun,
    K,
    T,
    c,
    r,
    q,
    sigma,
    barrier,
    dummy,
    sim,
    seed=111,
    antithetic=True,
    notional=10000.0,
    dtype="float64",
    pricer=None,
    bump_S_frac=0.01,
    bump_sigma_frac=0.01,
    bump_r_bp=1,
    gamma_smooth=True,
    gamma_mults=(0.5, 1.0, 2.0, 4.0, 8.0),
    gamma_agg="median",
    gamma_trim=0.25,
    sigma_fx=sigma_fx,
    rho_s_fx=rho_s_fx,
    r_f=r_f,
    quanto_on=True,
)

{'Price': 9298.447255355986,
 'Delta': 0.6690292233711974,
 'Gamma': 0.0002121238542301924,
 'Vega(1%)': -42.72766425404977,
 'Theta(1d)': 6.295742913063805,
 'Rho(1bp)': -0.9230802458696418,
 'Vega_FX(1%)': 0.0,
 'Corr(1pt)': -1.9190967113881925,
 'Rho_f(1bp)': 0.6198816110972984,
 'bump': {'eps_S_delta': 20.400000000000002,
  'eps_S_gamma_base': 68.0,
  'eps_sigma': 0.003,
  'eps_r': 0.0001,
  'CRN': True,
  'antithetic': True,
  'seed': 111,
  'gamma_smooth': True,
  'gamma_mults': (0.5, 1.0, 2.0, 4.0, 8.0),
  'gamma_agg': 'median',
  'quanto': {'sigma_fx': 0.1,
   'rho_s_fx': 0,
   'quanto_on': True,
   'bump_sigma_fx_frac': 0.01,
   'bump_rho': 0.005}}}

In [11]:
# 스텝다운 상품 평가 정보
S0=6800
kijun=6800
r=0.035
r_f=0.05
q=0
sigma, barrier, dummy, sim=0.3, 0.65, 0.132,10000
K=[0.95,0.95,0.95,0.90,0.90,0.85]
T=np.array([ 181,  365,  546,  730,  912, 1098])
c=np.array([1,2,3,4,5,6])*0.022
sigma_fx=0.1
rho_s_fx=-0.6

MC_Greeks_SD1_Quanto_GPU_CRN(
    S0,
    kijun,
    K,
    T,
    c,
    r,
    q,
    sigma,
    barrier,
    dummy,
    sim,
    seed=111,
    antithetic=True,
    notional=10000.0,
    dtype="float64",
    pricer=None,
    bump_S_frac=0.01,
    bump_sigma_frac=0.01,
    bump_r_bp=1,
    gamma_smooth=True,
    gamma_mults=(0.5, 1.0, 2.0, 4.0, 8.0),
    gamma_agg="median",
    gamma_trim=0.25,
    sigma_fx=sigma_fx,
    rho_s_fx=rho_s_fx,
    r_f=r_f,
    quanto_on=True,
)

{'Price': 9401.555116790181,
 'Delta': 0.47566075754556386,
 'Gamma': -4.474167934631623e-05,
 'Vega(1%)': -32.246904196375304,
 'Theta(1d)': 2.384236658368536,
 'Rho(1bp)': -0.9105513966414946,
 'Vega_FX(1%)': 11.489617980232651,
 'Corr(1pt)': -2.1505169488009397,
 'Rho_f(1bp)': 0.9496751582673824,
 'bump': {'eps_S_delta': 20.400000000000002,
  'eps_S_gamma_base': 68.0,
  'eps_sigma': 0.003,
  'eps_r': 0.0001,
  'CRN': True,
  'antithetic': True,
  'seed': 111,
  'gamma_smooth': True,
  'gamma_mults': (0.5, 1.0, 2.0, 4.0, 8.0),
  'gamma_agg': 'median',
  'quanto': {'sigma_fx': 0.1,
   'rho_s_fx': -0.6,
   'quanto_on': True,
   'bump_sigma_fx_frac': 0.01,
   'bump_rho': 0.005}}}